# Meeting 1: Classes as Objects and Python’s Object Model

## Instance and class attributes, namespaces and shadowing

In [26]:
class Student:
    count = 0  # Class attribute

    def __init__(self, name):
        self.name = name  # Instance attribute

david = Student('David')
john = Student('John')

'count' in david.__dict__, 'count' in type(david).__dict__
# (False, True)

Student.count, david.count

(0, 0)

In [27]:
class Student:
    count = 0  # Class attribute

    def __init__(self, name):
        self.name = name  # Instance attribute
        Student.count += 1

david = Student('David')

Student.count, david.count
# (1, 1)

(1, 1)

In [28]:
class Student:
    count = 0  # Class attribute

    def __init__(self, name):
        self.name = name  # Instance attribute
        self.count = self.count + 1
        # The above statement has two distinct operations:
        # - Reading self.count: Finds the class attribute Student.count,
        #   whose value is 0.
        # - Assigning self.count:
        #   Creates an instance attribute with value 1,
        #   shadowing Student.count.

david = Student('David')

Student.count, david.count
# (0, 1)

(0, 1)

In [29]:
'count' in Student.__dict__, 'count' in david.__dict__

(True, True)

In [30]:
del david.count

In [31]:
david.__dict__

{'name': 'David'}

In [32]:
'count' in Student.__dict__

True

## Dynmanic creation of classes, object and type

In [33]:
# create a class C using class statement
class C:
    x = 5

C, type(C), C.__bases__

(__main__.C, type, (object,))

In [34]:
# type is the default metaclass (class of classes).
# Every ordinary class is an instance of type.

In [35]:
type(2) is int       # True: 2 is an instance of int
type(True) is bool   # True: True is an instance of bool

True

In [36]:
# Use type to create a class dynamically.
# type is Python's default metaclass.
#
# name      = 'C'
# bases     = (object,)
# namespace = {'x': 5}

C = type('C', (object,), {'x': 5})

C, type(C), C.__bases__, C.__dict__

(__main__.C,
 type,
 (object,),
 mappingproxy({'x': 5,
               '__module__': '__main__',
               '__dict__': <attribute '__dict__' of 'C' objects>,
               '__weakref__': <attribute '__weakref__' of 'C' objects>,
               '__doc__': None}))

In [37]:
class Student:
    pass

class GraduateStudent(Student):
    pass

s = Student()
g = GraduateStudent()

type(s), type(2), type(g), isinstance(g, GraduateStudent)
# (Student, int, GraduateStudent, True)

issubclass(GraduateStudent, Student), isinstance(g, Student)
# (True, True)

type(Student), type(GraduateStudent)
# (type, type)

(type, type)

In [38]:
# In Python, classes are themselves objects, just like their instances.
isinstance(Student, object)       # True
isinstance(GraduateStudent, object)  # True

True

In [39]:
class C:
    pass

c = C()

In [40]:
# Class inheritance hierarchy in Python:
#
#                   object (__new__, __getattribute__)
#                 /   |    \
#                C  Student  type (__new__, for creating new classes)
#                     |
#               GraduateStudent
#
# object is the root of the inheritance hierarchy.
# type inherits from object and provides additional
# behavior for class objects.

Student.__bases__              # (object,)
issubclass(Student, object)     # True

GraduateStudent.__bases__      # (Student,)
type.__bases__                 # (object,)
object.__bases__               # ()

()

In [41]:
# object provides fundamental behavior for objects.
# type overrides or extends this behavior for class objects.

# Both object and type define these methods:
'__new__' in object.__dict__, '__new__' in type.__dict__
# (True, True)

'__init__' in object.__dict__, '__init__' in type.__dict__
# (True, True)

'__getattribute__' in object.__dict__, '__getattribute__' in type.__dict__
# (True, True)

# type defines additional behavior for class objects:
'__call__' in type.__dict__       # True
'__bases__' in type.__dict__      # True

True

In [42]:
# object.__new__ creates ordinary instances.
# type.__new__ overrides object.__new__ to create class objects.
#
# Both methods are defined directly in their respective classes.

object.__new__  # Default object creation
type.__new__    # Class creation

<function type.__new__(*args, **kwargs)>

In [43]:
########
class C(object):
    def __new__(cls, *args, **kwargs):
        print('C new', cls, cls.__mro__)

        # __new__ is implicitly a static method.
        # super().__new__ resolves to object.__new__.
        # We must explicitly pass cls to specify
        # which class's instance to create.
        return super().__new__(cls)

    def __init__(self):
        print('C init')

        # __init__ is an ordinary instance method.
        # super().__init__ is already bound to self.
        # Therefore, self is passed automatically.
        super().__init__()

c = C()

C new <class '__main__.C'> (<class '__main__.C'>, <class 'object'>)
C init


In [44]:
class D:
    pass

d = D()
c = C()

C new <class '__main__.C'> (<class '__main__.C'>, <class 'object'>)
C init


In [45]:
# __bases__ is provided by the metaclass, type,
# rather than stored in Student's own namespace.

'__bases__' in Student.__dict__        # False
'__bases__' in type(Student).__dict__  # True

Student.__bases__          # (object,)
GraduateStudent.__bases__  # (Student,)

########
'__bases__' in GraduateStudent.__dict__        # False
'__bases__' in type(GraduateStudent).__dict__  # True

True

In [46]:
########
# __bases__ is a data descriptor defined in type.

d = type.__dict__['__bases__']

type(d)
# <class 'getset_descriptor'>

getset_descriptor

In [47]:
# Instance-class relationships
isinstance(david, Student)   # True
isinstance(2, int)           # True
isinstance(david, object)    # True

# Classes are also objects
isinstance(Student, object)  # True, Student is an object
isinstance(Student, type)    # True, Student is a class
isinstance(object, type)     # True, object is a class
isinstance(type, object)     # True, type is an object

########
isinstance(type, type)       # True, type is itself a class
isinstance(object, object)   # True, object is also an object

True

In [48]:
# Class inheritance (subclass) relationships
issubclass(type, object)     # True, type inherits from object
issubclass(Student, object)  # True, Student inherits from object
issubclass(object, type)     # False! object does not inherit from type

False

## Instance, class, and static methods

In [49]:
class Student:
    def __init__(self, name, major):
        self.name = name

        if not Student.valid_major(major):
            raise ValueError('Invalid major')

        self.major = major

    # Instance method: self refers to the instance
    def introduce(self):
        print('self =', self)
        print(f'I am {self.name}. I major in {self.major}.')

    # Class method: cls refers to the class
    @classmethod
    def from_str(cls, text):
        # print('cls =', cls)
        name, major = text.split(',')
        return cls(name, major)

    # Static method: receives neither self nor cls automatically
    @staticmethod
    def valid_major(major):
        return major in {'CS', 'ADS', 'Math'}

david = Student('David', 'Math')

# Instance method: automatically binds david as self
david.introduce()

# Class method: automatically binds Student as cls
bill = Student.from_str('Bill,CS')

# A class method can also be called through an instance.
# cls is still Student, not david.
john = david.from_str('John,ADS')

# Static method: no self or cls is automatically supplied
Student.valid_major('ADS')  # True
david.valid_major('math')   # False

self = <__main__.Student object at 0x7c8c9417fa10>
I am David. I major in Math.


False

## properties

In [50]:
class Student:
    def __init__(self, name, major, age):
        self.name = name
        self.major = major
        self.age = age  # Automatically invokes the property setter

    @property
    def age(self):
        print('age getter')
        return self._age

    @age.setter
    def age(self, age):
        print('age setter')
        if type(age) is not int or age < 0:
            raise ValueError('Invalid age')

        self._age = age  # Store the validated value


david = Student('David', 'CS', 5)  # Invokes age setter
david.age                        # Invokes age getter

age setter
age getter


5

# Meeting 2: Descriptors and Python’s Attribute Lookup

## Functions, descriptors, and bound Methods

In [65]:
class C:
    def f(self, x):
        return x * 2

c = C()

# c.f(3)

# Conceptually, Python first evaluates:
m = c.f

# Then calls:
m(3)

6

In [69]:
print(m) # m is a bound method

<bound method C.f of <__main__.C object at 0x7c8c87eb0050>>


In [77]:
# How does Python create the bound method?

# Because f is a function stored in C and functions are descriptors,
# Python conceptually does:

f = C.__dict__['f']

# Python calls f.__get__ to bind f to the instance c.
m = f.__get__(c, C)

# m is now a bound method:
type(m)  # method

m.__self__ is c   # True
m.__func__ is f   # True

# Therefore:
m(3)

# effectively calls:
f(c, 3)

6

In [76]:
# Note: functions are non-data descriptors that define __get__
# but not __set__ or __delete__.

hasattr(f, '__get__'), hasattr(f, '__set__'), hasattr(f, '__delete__')
# (True, False, False)

(True, False, False)

In [79]:
# Looking up f through an instance returns a bound method,
# while looking up f through the class returns the function itself.

type(c.f), type(C.f)
# (<class 'method'>, <class 'function'>)

f = C.__dict__['f']

# Conceptually:
print(c.f)  # f.__get__(c, C)     -> bound method
print(C.f)  # f.__get__(None, C)  -> function

<bound method C.f of <__main__.C object at 0x7c8c87eb0050>>
<function C.f at 0x7c8c87e989a0>


In [ ]:
# A descriptor's __get__ method receives:
#
#   __get__(instance, owner)
#
# instance: the instance through which the attribute is accessed
#           None if accessed through the class
#
# owner:    the class through which the attribute is accessed

In [ ]:
# A descriptor is an object stored as a class attribute
# that defines special methods controlling attribute access.

# Data descriptor:
#   defines __get__ and __set__ and/or __delete__
#   has higher priority than instance attributes (see below)
#
# Non-data descriptor:
#   defines __get__ only
#   has lower priority than instance attributes (see below)

## Attribute lookup (`__getattribute__`)
* How does Python find f in the first place?

In [90]:
class C():
    def f(self, x):
        return x * 2

    # called for EVERY attribute lookup
    def __getattribute__(self, name): # override object.__getattribute__
        print('C getattribute', name)
        return super().__getattribute__(name) # bound method for c.f
        #return object.__getattribute__(self, name)

    # called only if normal attribute lookup fails
    def __getattr__(self, name):
        print('C getattr')
        raise AttributeError(name)

        # Note that object does not define __getattr__, so this will fail:
        #return super().__getattr__(name)

    # Called for EVERY instance attribute assignment
    def __setattr__(self, name, value):
        print('C setattr', name, value)
        super().__setattr__(name, value)

c = C()

# Attribute assignment calls __setattr__.
c.x = 5
# Conceptually:
# C.__setattr__(c, 'x', 5)

# Attribute lookup calls __getattribute__.
c.x
# Conceptually:
# C.__getattribute__(c, 'x')

C setattr x 5
C getattribute x


5

In [87]:
# c.y   # missing: __getattribute__, then __getattr__, then AttributeError

In [ ]:
# Conceptual attribute lookup for c.x
# C = type(c)

# First, search for x in C.__mro__.
# Let attr be the first matching class attribute.

# 1. If attr is a data descriptor:
#       return attr.__get__(c, C)

# 2. If x is in c.__dict__:
#       return c.__dict__['x']

# 3. If attr is a non-data descriptor:
#       return attr.__get__(c, C)

# 4. If attr is an ordinary class attribute:
#       return attr

# 5. Otherwise:
#       raise AttributeError
#       # __getattr__ may be invoked as a fallback

In [89]:
# Instance attributes have higher priority than non-data descriptors.

class C:
    def f(self, x):
        return x * 2

c = C()

c.f(3)       # 6

c.f = 5
c.f          # 5

# An instance attribute overrides a non-data descriptor
# such as an ordinary Python function.
c.__dict__

{'f': 5}

In [95]:
# A property is a data descriptor and therefore has
# higher lookup priority than the instance dictionary.
class C:
    @property
    def age(self):
        print('C age property...')
        return 5

c = C()

# Manually put an 'age' attribute into the instance dictionary.
c.__dict__['age'] = -5

c.__dict__
# {'age': -5}

# But the property has higher lookup priority.
c.age
# C age property...
# 5

C age property...


5

In [97]:
p = C.__dict__['age']
type(p), p

(property, <property at 0x7c8c87eba070>)

In [100]:
hasattr(p, '__get__'), hasattr(p, '__set__'), hasattr(p, '__delete__')

(True, True, True)

## Instance, Class, and Static Methods: Binding Behavior

In [105]:
class Student:
    def __init__(self, name, major, age):
        self.name = name
        self.major = major
        self.age = age   # calls the age setter

    def introduce(self):
        print(f'I am {self.name}.')

    @classmethod
    def from_str(cls, text):
        name, major, age = text.split(',')
        return cls(name, major, int(age))

    @staticmethod
    def valid_major(major):
        return major in {"CS", "Math"}

    # The following is equivalent to using @property
    def get_age(self):
        return self._age

    age = property(get_age)

    @age.setter
    def age(self, age):
        self._age = age

In [106]:
david = Student("David", "CS", 25)
david.__dict__

{'name': 'David', 'major': 'CS', '_age': 25}

In [129]:
# Accessing an instance method through an instance
# creates a bound method.
m = david.introduce

m.__self__, m.__func__

(<__main__.Student at 0x7c8c9417fb60>,
 <function __main__.Student.introduce(self)>)

In [137]:
# Accessing an instance method through the class
# returns the function itself.
Student.introduce is Student.__dict__['introduce']
# True

True

In [133]:
# A class method is bound to the class,
# even when accessed through an instance.
m = david.from_str
print(m.__self__, m.__func__)
# Student, <function Student.from_str ...>

# A class method is also bound to the class
# when accessed through the class.
m = Student.from_str
print(m.__self__, m.__func__)
# Student, <function Student.from_str ...>

<class '__main__.Student'> <function Student.from_str at 0x7c8c87ed7ce0>
<class '__main__.Student'> <function Student.from_str at 0x7c8c87ed7ce0>


In [139]:
# A static method is not bound to the instance.
# Accessing it through an instance returns the underlying function.
print(david.valid_major)

# A static method is also not bound to the class.
# Accessing it through the class returns the same underlying function.
print(Student.valid_major)

david.valid_major is Student.valid_major   # True

<function Student.valid_major at 0x7c8c87ed7c40>
<function Student.valid_major at 0x7c8c87ed7c40>


True

In [176]:
# There is no __self__:
m = david.valid_major

hasattr(m, '__self__')
# False

False

In [ ]:
# Contrast all three method types:

# Instance method: binds the instance
david.introduce.__self__ is david          # True

# Class method: binds the class
david.from_str.__self__ is Student         # True
Student.from_str.__self__ is Student       # True

# Static method: no binding
david.valid_major is Student.valid_major   # True

## Custom descriptor

In [141]:
class D:
    def __get__(self, instance, owner):
        print('getter')

        # If accessed through the class, e.g. C.age
        if instance is None:
            return self

        # If accessed through an instance, e.g. c.age
        return instance._age

    def __set__(self, instance, value):
        print('setter')
        instance._age = value


class C:
    age = D()  # data descriptor: defines __get__ and __set__


c = C()

# Attribute assignment finds the data descriptor C.__dict__['age']
# and calls its __set__ method.
c.age = 5
# conceptually:
# C.__dict__['age'].__set__(c, 5)

# __set__ stores:
c.__dict__
# {'_age': 5}


# Attribute lookup finds the data descriptor
# and calls its __get__ method.
c.age
# conceptually:
# C.__dict__['age'].__get__(c, C)
# getter
# 5

setter
getter


5

In [142]:
attr = C.__dict__['age']

# Access through an instance:
# c is an instance of C, so lookup uses C.__getattribute__.
c.age
# conceptually:
D.__get__(attr, c, C)


# Access through the class:
# C is itself an instance of type, so lookup uses type.__getattribute__.
C.age
# conceptually:
D.__get__(attr, None, C)  # returns the descriptor itself

getter
getter
getter
getter


In [144]:
# Optional: conceptual model of a bound method.
# This example is for understanding how method binding works.
# You are not expected to implement BoundMethod on an exam.

class BoundMethod:
    def __init__(self, instance, func):
        self.__self__ = instance
        self.__func__ = func

    def __call__(self, *args, **kwargs):
        return self.__func__(self.__self__, *args, **kwargs)

In [146]:
class C:
    def f(self, x):
        return x * 2

c = C()

f = C.__dict__['f']

m = BoundMethod(c, f)

m(3)   # 6

6

## Methods and inheritance
* Practiced in homework 7

In [157]:
class Student:
    def __init__(self, name, major, age):
        self.name = name
        self.major = major
        self.age = age   # calls the age setter

    def introduce(self):
        print(f'I am {self.name}.')

    @classmethod
    def from_str(cls, text):
        name, major, age = text.split(',')
        return cls(name, major, int(age))

    @staticmethod
    def valid_major(major):
        return major in {"CS", "Math"}

    def get_age(self): # getter
        print('getter', self)
        return self._age

    age = property(get_age)

    @age.setter
    def age(self, age):
        print('setter', self, age)
        self._age = age

class GraduateStudent(Student):
    pass

g = GraduateStudent('Bill', 'ADS', 25)
g


setter <__main__.GraduateStudent object at 0x7c8c87eb0590> 25


In [158]:
# properties are inherited and still bind to the subclass instance
g.age

getter <__main__.GraduateStudent object at 0x7c8c87eb0590>


25

In [162]:
# g.age

# Conceptually, Python finds the inherited property in Student:
p = Student.__dict__['age']

# Then calls its __get__ with g as the instance,
# which in turn calls the getter:
#     Student.get_age(g)
p.__get__(g, GraduateStudent)

# Note: a property does not normally return a bound method.
# It uses its getter function to compute and return a value.

# Note: the descriptor was defined in Student,
# but it binds to g, an instance of GraduateStudent.

getter <__main__.GraduateStudent object at 0x7c8c87eb0590>


25

In [166]:
# Since introduce is inherited from Student,
# accessing it through g creates a bound method whose instance is g
# and whose underlying function is the function defined in Student.
m = g.introduce

m.__self__ is g
# True

m.__func__ is Student.__dict__['introduce']
# True

True

In [168]:
# Even though from_str is defined in Student,
# it is inherited by GraduateStudent.
# When accessed through g, the class method binds to the class of g,
# which is GraduateStudent.
m = g.from_str
m.__self__ is GraduateStudent, m.__func__

(True, <function __main__.Student.from_str(cls, text)>)

In [170]:
m = GraduateStudent.from_str
m.__self__ is GraduateStudent, m.__func__

(True, <function __main__.Student.from_str(cls, text)>)

In [173]:
# Static methods do not bind an instance or a class.

g.valid_major, GraduateStudent.valid_major, Student.valid_major


(<function __main__.Student.valid_major(major)>,
 <function __main__.Student.valid_major(major)>,
 <function __main__.Student.valid_major(major)>)

In [174]:
# All return the underlying function:
g.valid_major is GraduateStudent.valid_major
# True

GraduateStudent.valid_major is Student.valid_major
# True

True

In [177]:
# There is no __self__:
m = g.valid_major

type(m)
# function

hasattr(m, '__self__')
# False

False